In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [10]:
#self attentiom

class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention,self).__init__()
        self.embed_size=embed_size

        self.values= nn.Linear(embed_size,embed_size)
        self.keys= nn.Linear(embed_size, embed_size)
        self.queries= nn.Linear(embed_size,embed_size)

    def forward(self, x):
        V=self.values(x)
        K=self.values(x)
        Q=self.queries(x)

        #attention score
        energy=torch.matmul(Q,K.transpose(-1,-2))
        attention= torch.softmax(energy/(self.embed_size **0.5), dim=-1)
        out=torch.matmul(attention,V)
        return out

In [11]:
#transformer block

class TransformerBlock(nn.Module):
    def __init__(self, embed_size):
        super(TransformerBlock, self).__init__()

        self.attention=SelfAttention(embed_size)
        self.norm1= nn.LayerNorm(embed_size)

        self.feed_forward= nn.Sequential(
            nn.Linear(embed_size, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size,embed_size)
        )
        self.norm2= nn.LayerNorm(embed_size)

    def forward(self,x):
        #attentiaon +residual
        attn=self.attention(x)
        x=self.norm1(attn+x)

        #feed forward+residual
        forward=self.feed_forward(x)
        out=self.norm2(forward+x)

        return out
        

In [12]:
#simple transformer model

class SimpleTransformer(nn.Module):
    def __init__(self,vocab_size,embed_size):
        super(SimpleTransformer, self).__init__()

        self.embedding=nn.Embedding(vocab_size, embed_size)
        self.transformer=TransformerBlock(embed_size)
        self.fc_out=nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x=self.embedding(x)
        x=self.transformer(x)
        out= self.fc_out(x)
        return out 

In [14]:
#example usage

model= SimpleTransformer(vocab_size=1000, embed_size=64)

x = torch.randint(0, 1000, (2, 5)) #batch=2, sequence len=5
output=model(x)

print(output.shape)

torch.Size([2, 5, 1000])
